In [1]:
import json
import numpy as np

In [2]:
# Завантаження параметрів моделі з JSON файлів
def load_weights(filename):
    with open(filename, 'r') as f:
        return np.array(json.load(f))

embedding = load_weights("embedding.weight.json")
U_h_bias = load_weights("U_h.bias.json")
U_h_weight = load_weights("U_h.weight.json")
W_h_bias = load_weights("W_h.bias.json")
W_h_weight = load_weights("W_h.weight.json")
W_y_weight = load_weights("W_y.weight.json")

In [3]:
# Завантаження словника
with open("vocab.json", 'r') as f:
    vocab_list = json.load(f)
    
# Створення оберненого словника для жадібного декодування
# Створення словника та оберненого словника
vocab = {char: idx for idx, char in enumerate(vocab_list)}
inv_vocab = {idx: char for char, idx in vocab.items()}

# Параметри моделі
embedding_dim = 96
hidden_dim = 160
vocab_size = 133

In [4]:
# Функція для forward-pass
def forward_pass(message, max_length=100):
    # Початковий стан прихованого шару
    h_t = np.zeros((hidden_dim,))
    decoded_message = []
    
    # Обробка першого символу - початкового токена
    x_t = embedding[vocab["["]]
    
    for _ in range(max_length):
        # Розрахунок поточного стану h_t
        h_t = np.tanh(W_h_weight @ x_t + W_h_bias + U_h_weight @ h_t + U_h_bias)
        
        # Вихідний шар
        y_t = W_y_weight @ h_t
        predicted_token = np.argmax(y_t)
        
        # Декодування символу
        decoded_char = inv_vocab[predicted_token]
        
        # Зупинка, якщо досягнуто кінцевий токен
        if decoded_char == "]":
            break
        
        decoded_message.append(decoded_char)
        
        # Оновлення x_t для наступної ітерації
        x_t = embedding[predicted_token]
    
    return "".join(decoded_message)

In [5]:
# Запуск forward-pass і вивід результату
message = forward_pass("[")
print("Закодоване повідомлення:", message)

Закодоване повідомлення: Нарди грати будеш?
